In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
!pip install netCDF4 torch-geometric scipy -q

# verify
import netCDF4
import torch_geometric
from scipy.stats import pearsonr
print('netCDF4:', netCDF4.__version__)
print('torch_geometric:', torch_geometric.__version__)
print('All dependencies OK')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 71.1 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 60.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 65.6 MB/s eta 0:00:00
netCDF4: 1.7.4
torch_geometric: 2.7.0
All dependencies OK


In [15]:
import os, gc
import numpy as np
import netCDF4 as nc
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# ── Paths ─────────────────────────────────────────────────────
BASE     = '/kaggle/input/datasets/divyanshyecho/enso-ham2019-dataset'
CKPT_DIR = '/kaggle/working/cp4b_checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)

CMIP5_INPUT = f'{BASE}/CMIP5.input.36mn.1861_2001.nc'
CMIP5_LABEL = f'{BASE}/CMIP5.label.nino34.12mn_3mv.1863_2003.nc'
SODA_INPUT  = f'{BASE}/SODA.input.36mn.1871_1970.nc'
SODA_LABEL  = f'{BASE}/SODA.label.nino34.12mn_3mv.1873_1972.nc'
GODAS_INPUT = f'{BASE}/GODAS.input.36mn.1980_2015.nc'
GODAS_LABEL = f'{BASE}/GODAS.label.12mn_3mv.1982_2017.nc'

# ── Config ────────────────────────────────────────────────────
LEAD       = 2      # n=3 month lead (0-indexed)
N_VARS     = 2      # SST + t300
N_MONTHS   = 36     # full 36 month window
IN_FEATS   = N_VARS * N_MONTHS   # 72 features per node
EPOCHS     = 150
BATCH_SIZE = 32     # reduced from 64 — 72 features uses more memory

SEEDS = [42, 123, 456, 789]

def set_seed(seed):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)
print(f'IN_FEATS: {IN_FEATS}')
print('Setup complete.')

Device: cuda
IN_FEATS: 72
Setup complete.


In [9]:
# ── Ocean mask ────────────────────────────────────────────────
ds_soda  = nc.Dataset(SODA_INPUT)
sst_all  = np.array(ds_soda.variables['sst'][:, 0, :, :])   # (100, 24, 72)
t300_all = np.array(ds_soda.variables['t300'][:, 0, :, :])  # (100, 24, 72)
ds_soda.close()

land_mask  = (sst_all == 0.0).all(axis=0) & \
             (t300_all == 0.0).all(axis=0)   # (24, 72)
ocean_mask = ~land_mask
ocean_idx  = np.where(ocean_mask.flatten())[0]
N_NODES    = int(ocean_mask.sum())
print(f'Ocean nodes: {N_NODES}  |  Land nodes: {24*72 - N_NODES}')

# ── Lat/lon for static node features ─────────────────────────
ds_tmp   = nc.Dataset(SODA_INPUT)
lats     = np.array(ds_tmp.variables['lat'][:])   # (24,)
lons     = np.array(ds_tmp.variables['lon'][:])   # (72,)
ds_tmp.close()

lat_grid = np.repeat(lats[:, None], 72, axis=1).flatten()[ocean_idx]
lon_grid = np.repeat(lons[None, :], 24, axis=0).flatten()[ocean_idx]

# ── Load function — full 36 months → 72 features per node ────
def load_dataset(input_file, label_file, sst_var='sst'):
    ds  = nc.Dataset(input_file)
    ds2 = nc.Dataset(label_file)

    sst  = np.array(ds.variables[sst_var][:]).astype(np.float32)  # (N, 36, 24, 72)
    t300 = np.array(ds.variables['t300'][:]).astype(np.float32)   # (N, 36, 24, 72)

    # Stack vars → (N, 2, 36, 24, 72)
    X = np.stack([sst, t300], axis=1)
    X = np.nan_to_num(X, nan=0.0)

    # Normalise per variable per spatial location
    mean = X.mean(axis=(0, 2), keepdims=True)  # (1, 2, 1, 24, 72)
    std  = X.std(axis=(0, 2),  keepdims=True) + 1e-6
    X    = (X - mean) / std

    # Filter to ocean nodes + flatten to 72 features
    # (N, 2, 36, 24, 72) → (N, 2, 36, 1728) → ocean → (N, 2, 36, N_NODES)
    N = X.shape[0]
    X = X.reshape(N, 2, 36, -1)[:, :, :, ocean_idx]  # (N, 2, 36, N_NODES)

    # → (N, N_NODES, 72)
    X = X.transpose(0, 3, 1, 2).reshape(N, N_NODES, -1)

    labels = np.array(ds2.variables['pr'][:]).astype(np.float32)
    ds.close()
    ds2.close()
    return X, labels

# ── Load all datasets ─────────────────────────────────────────
print('Loading CMIP5...')
X_cmip5, y_cmip5_raw = load_dataset(CMIP5_INPUT, CMIP5_LABEL, sst_var='sst1')
y_cmip5 = y_cmip5_raw[:, LEAD, 0, 0]
print(f'  X={X_cmip5.shape}  y={y_cmip5.shape}')

print('Loading SODA...')
X_soda, y_soda_raw = load_dataset(SODA_INPUT, SODA_LABEL, sst_var='sst')
y_soda = y_soda_raw[:, LEAD, 0, 0]
print(f'  X={X_soda.shape}  y={y_soda.shape}')

print('Loading GODAS...')
X_godas, y_godas_raw = load_dataset(GODAS_INPUT, GODAS_LABEL, sst_var='sst')
y_godas = y_godas_raw[:, LEAD, 0, 0]
print(f'  X={X_godas.shape}  y={y_godas.shape}')

print('\nExpected shapes:')
print('  CMIP5:  (2961, 1393, 72)')
print('  SODA:   (100,  1393, 72)')
print('  GODAS:  (36,   1393, 72)')


# ── Static node features — computed while X_soda is in memory ─
soda_mean_sst  = X_soda[:, :, :36].mean(axis=(0, 2))  # (N_NODES,)
soda_mean_t300 = X_soda[:, :, 36:].mean(axis=(0, 2))  # (N_NODES,)

lat_norm = (lat_grid - lat_grid.mean()) / (lat_grid.std() + 1e-6)
lon_norm = (lon_grid - lon_grid.mean()) / (lon_grid.std() + 1e-6)

X_static = np.stack([soda_mean_sst,
                     soda_mean_t300,
                     lat_norm,
                     lon_norm], axis=1).astype(np.float32)  # (N_NODES, 4)

X_static_tensor = torch.tensor(X_static, dtype=torch.float32)
print(f'Static node features: {X_static.shape}')  # expect (1393, 4)

Ocean nodes: 1393  |  Land nodes: 335
Loading CMIP5...
  X=(2961, 1393, 72)  y=(2961,)
Loading SODA...
  X=(100, 1393, 72)  y=(100,)
Loading GODAS...
  X=(36, 1393, 72)  y=(36,)

Expected shapes:
  CMIP5:  (2961, 1393, 72)
  SODA:   (100,  1393, 72)
  GODAS:  (36,   1393, 72)
Static node features: (1393, 4)


In [10]:
# ── Graph construction ────────────────────────────────────────
def make_graphs(X, y):
    graphs = []
    y_t = torch.tensor(y, dtype=torch.float32)
    for i in range(len(X)):
        graphs.append(Data(
            x = torch.tensor(X[i], dtype=torch.float32),
            y = y_t[i]
        ))
    return graphs

print('Building graphs...')
graphs_cmip5 = make_graphs(X_cmip5, y_cmip5)
graphs_soda  = make_graphs(X_soda,  y_soda)
graphs_godas = make_graphs(X_godas, y_godas)

graphs_train = graphs_cmip5 + graphs_soda

print(f'CMIP5:  {len(graphs_cmip5)} graphs')
print(f'SODA:   {len(graphs_soda)}  graphs')
print(f'Train:  {len(graphs_train)} graphs total')
print(f'GODAS:  {len(graphs_godas)} graphs (test only)')

del X_cmip5, X_soda, X_godas
gc.collect()
print('Memory freed.')

Building graphs...
CMIP5:  2961 graphs
SODA:   100  graphs
Train:  3061 graphs total
GODAS:  36 graphs (test only)
Memory freed.


In [11]:
# ── Structure Learning Module ─────────────────────────────────
class GraphStructureLearner(nn.Module):
    def __init__(self, in_feats, d2=16, alpha1=0.1, alpha2=2.0,
                 n_nodes=1393, x_static=None, device='cuda'):
        super().__init__()
        self.num_nodes   = n_nodes
        self.alpha1      = alpha1
        self.alpha2      = alpha2
        self.device      = device

        self.lin1 = nn.Linear(in_feats, d2)
        self.lin2 = nn.Linear(in_feats, d2)

        self.static_feat = x_static.float().to(device)

    def forward(self):
        nodevec1 = torch.tanh(self.alpha1 * self.lin1(self.static_feat))
        nodevec2 = torch.tanh(self.alpha1 * self.lin2(self.static_feat))

        # Soft adjacency — fully differentiable
        adj = torch.sigmoid(self.alpha2 * nodevec1 @ nodevec2.T)

        # Row normalise
        adj = adj / (adj.sum(dim=1, keepdim=True) + 1e-6)

        # Self loops
        adj = adj + torch.eye(self.num_nodes, device=adj.device) * 0.5

        return adj  # (N_NODES, N_NODES)


# ── GCN Layer ─────────────────────────────────────────────────
class GCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim, residual=False):
        super().__init__()
        self.linear   = nn.Linear(in_dim, out_dim)
        self.bn       = nn.BatchNorm1d(out_dim)
        self.residual = residual and (in_dim == out_dim)

    def forward(self, x, A, batch):
        batch_size = batch.max().item() + 1
        out_list   = []

        for g in range(batch_size):
            mask  = (batch == g)
            x_g   = x[mask]
            out_g = A @ x_g
            out_list.append(out_g)

        out = torch.cat(out_list, dim=0)
        out = self.linear(out)
        out = self.bn(out)
        out = F.elu(out)

        if self.residual:
            out = out + x
        return out


# ── Full Graphino Model ───────────────────────────────────────
class Graphino(nn.Module):
    def __init__(self, in_feats=72, n_nodes=1393,
                 layer_dims=[250, 250],
                 pooling='mean',
                 weight_decay=1e-4,
                 x_static=None):
        super().__init__()
        self.n_nodes    = n_nodes
        self.pooling    = pooling
        self.layer_dims = layer_dims

        self.structure = GraphStructureLearner(
            in_feats = x_static.shape[1],   # 4
            d2       = 16,
            alpha1   = 0.1,
            alpha2   = 2.0,
            n_nodes  = n_nodes,
            x_static = x_static,
            device   = str(device)
        )

        self.gcn_layers = nn.ModuleList()
        prev_dim = in_feats
        for dim in layer_dims:
            residual = (prev_dim == dim)
            self.gcn_layers.append(GCNLayer(prev_dim, dim, residual=residual))
            prev_dim = dim

        jk_dim = sum(layer_dims)
        mlp_in = jk_dim * 2 if pooling == 'mean+sum' else jk_dim

        self.mlp = nn.Sequential(
            nn.Linear(mlp_in, 128),
            nn.BatchNorm1d(128),
            nn.ELU(),
            nn.Linear(128, 1)
        )

    def forward(self, data):
        x     = data.x
        batch = data.batch

        A = self.structure()

        layer_outs = []
        for gcn in self.gcn_layers:
            x = gcn(x, A, batch)
            layer_outs.append(x)

        x_jk = torch.cat(layer_outs, dim=-1)

        batch_size = batch.max().item() + 1
        jk_dim     = x_jk.shape[-1]
        counts     = torch.bincount(batch).float().unsqueeze(1)

        pooled_sum = torch.zeros(batch_size, jk_dim, device=x.device)
        pooled_sum.scatter_add_(
            0, batch.unsqueeze(1).expand(-1, jk_dim), x_jk
        )
        pooled_mean = pooled_sum / counts

        if self.pooling == 'mean+sum':
            g = torch.cat([pooled_mean, pooled_sum], dim=-1)
        else:
            g = pooled_mean

        return self.mlp(g).squeeze(-1)


# ── Ensemble configs ──────────────────────────────────────────
ENSEMBLE_CONFIGS = [
    dict(layer_dims=[250, 100],      pooling='mean',     weight_decay=1e-6),
    dict(layer_dims=[250, 250],      pooling='mean',     weight_decay=1e-6),
    dict(layer_dims=[200, 200, 200], pooling='mean+sum', weight_decay=1e-3),
    dict(layer_dims=[250, 250, 250], pooling='mean+sum', weight_decay=1e-4),
]

# ── Sanity check ──────────────────────────────────────────────
set_seed(42)
test_model = Graphino(
    in_feats   = IN_FEATS,
    n_nodes    = N_NODES,
    layer_dims = [250, 250],
    pooling    = 'mean',
    x_static   = X_static_tensor
).to(device)

n_params = sum(p.numel() for p in test_model.parameters())
print(f'Model params: {n_params:,}')

test_loader = DataLoader(graphs_godas[:2], batch_size=2)
batch_test  = next(iter(test_loader)).to(device)
with torch.no_grad():
    out = test_model(batch_test)
print(f'Forward pass output shape: {out.shape}')
print('Model OK')
del test_model
gc.collect()

Model params: 146,673
Forward pass output shape: torch.Size([2])
Model OK


108

In [16]:
def train_one_model(layer_dims, pooling, weight_decay,
                    seed, model_id, resume=True):
    set_seed(seed)

    model = Graphino(
        in_feats   = IN_FEATS,
        n_nodes    = N_NODES,
        layer_dims = layer_dims,
        pooling    = pooling,
        x_static   = X_static_tensor
    ).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr           = 0.0005,
        weight_decay = weight_decay
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode     = 'max',
        factor   = 0.5,
        patience = 10,
        min_lr   = 1e-4
    )

    criterion = nn.MSELoss()

    train_loader = DataLoader(graphs_train, batch_size=BATCH_SIZE,
                              shuffle=True)
    godas_loader = DataLoader(graphs_godas, batch_size=36,
                              shuffle=False)

    ckpt_path   = f'{CKPT_DIR}/model{model_id}_seed{seed}_latest.pt'
    best_path   = f'{CKPT_DIR}/model{model_id}_seed{seed}_best.pt'
    start_epoch = 1
    best_cc     = -999
    history     = {'train_loss': [], 'godas_cc': []}

    if resume and os.path.exists(ckpt_path):
        print(f'  Resuming: {ckpt_path}')
        ckpt = torch.load(ckpt_path, map_location=device,
                          weights_only=False)
        model.load_state_dict(ckpt['model_state'])
        optimizer.load_state_dict(ckpt['optimizer_state'])
        start_epoch = ckpt['epoch'] + 1
        best_cc     = ckpt['best_cc']
        history     = ckpt['history']
        print(f'  Resuming from epoch {start_epoch} | '
              f'best CC: {best_cc:.4f}')
    else:
        print(f'  Starting fresh — model{model_id} seed{seed}')

    for epoch in range(start_epoch, EPOCHS + 1):

        # ── Train ─────────────────────────────────────────────
        model.train()
        total_loss = 0.0
        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            pred = model(batch)
            loss = criterion(pred, batch.y)
            if torch.isnan(loss):
                print(f'  WARNING: NaN at epoch {epoch}')
                continue
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        history['train_loss'].append(avg_loss)

        # ── Evaluate ──────────────────────────────────────────
        model.eval()
        preds, targets = [], []
        with torch.no_grad():
            for batch in godas_loader:
                batch = batch.to(device)
                preds.extend(model(batch).cpu().numpy())
                targets.extend(batch.y.cpu().numpy())

        preds   = np.array(preds)
        targets = np.array(targets)
        cc, _   = pearsonr(preds, targets)
        history['godas_cc'].append(cc)

        scheduler.step(cc)

        # ── Save checkpoint ───────────────────────────────────
        torch.save({
            'epoch'          : epoch,
            'model_state'    : model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'best_cc'        : best_cc,
            'history'        : history,
        }, ckpt_path)

        if cc > best_cc:
            best_cc = cc
            torch.save(model.state_dict(), best_path)

        if epoch % 10 == 0:
            current_lr = optimizer.param_groups[0]['lr']
            print(f'  Epoch {epoch:3d}/{EPOCHS} | '
                  f'Loss {avg_loss:.4f} | '
                  f'GODAS CC {cc:.4f} | '
                  f'Best CC {best_cc:.4f} | '
                  f'LR {current_lr:.6f}')

    print(f'\n  Done. Best GODAS CC: {best_cc:.4f}')
    return model, history, best_cc


# ── Run development model ─────────────────────────────────────
print('='*55)
print('CP4b — Development run: Model 2, seed 42')
print('='*55)

model, history, best_cc = train_one_model(
    layer_dims   = [250, 250],
    pooling      = 'mean',
    weight_decay = 1e-6,
    seed         = 42,
    model_id     = 2,
    resume       = True
)

CP4b — Development run: Model 2, seed 42
  Resuming: /kaggle/working/cp4b_checkpoints/model2_seed42_latest.pt
  Resuming from epoch 101 | best CC: 0.8595
  Epoch 110/150 | Loss 0.1138 | GODAS CC 0.8388 | Best CC 0.8605 | LR 0.000250
  Epoch 120/150 | Loss 0.1085 | GODAS CC 0.8478 | Best CC 0.8605 | LR 0.000125
  Epoch 130/150 | Loss 0.0996 | GODAS CC 0.8403 | Best CC 0.8605 | LR 0.000100
  Epoch 140/150 | Loss 0.0990 | GODAS CC 0.8164 | Best CC 0.8605 | LR 0.000100
  Epoch 150/150 | Loss 0.0967 | GODAS CC 0.8290 | Best CC 0.8605 | LR 0.000100

  Done. Best GODAS CC: 0.8605
